# Web Scraping and Text Processing

This notebook uses Requests, Beautiful Soup, regular expressions, and pandas to extract structured article metadata from NPR archive pages.

## Beautiful Soup

`pandas.read_html` works well for HTML tables. Less structured pages require an HTML parser such as Beautiful Soup.

In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import pandas as pd

The first workflow retrieves the [NPR Technology archive](https://www.npr.org/sections/technology/archive) and inspects its HTML structure.

In [ ]:
url = 'https://www.npr.org/sections/technology/archive'
html = requests.get(url).text

soup = BeautifulSoup(html, 'html5lib') # parse the html

print(soup.prettify()) # print a nicely formatted look at the html

Each archive entry is contained in an `<article>` element. Collecting those elements creates the set of stories to parse.

In [ ]:
articles = soup.find_all('article')
len(articles)  # How many articles did we get?

25

In [ ]:
articles[0] # Look at the content of the first article

Article metadata appears within nested `<div>` elements. The story title is extracted from the heading inside the `item-info` container.

In [ ]:
title = articles[0].find("div", "item-info").h2.text
title

'How audiobooks are made'

The `item-info` container also includes a short article description or teaser.

In [ ]:
teaser = articles[0].find("div", "item-info").p.text
teaser

'July 9, 2023 • From the podcast Nerdette from WBEZ: How can a "book on tape" bring a text to life?'

The paragraph element contains both the publication date and teaser, which are extracted separately.

In [ ]:
date = articles[0].find("div", "item-info").p.span.text
date

'July 9, 2023 • '

In [ ]:
teaser = teaser.replace(date, '')
teaser

'From the podcast Nerdette from WBEZ: How can a "book on tape" bring a text to life?'

A regular expression separates the textual date into month, day, and year capture groups.

In [ ]:
import re

dmatch = re.match(r'([a-zA-Z]+) (\d+), (\d+)', date)
print('The whole match = ', dmatch.group(0))
month = dmatch.group(1)
print('Month = ', month)
day = dmatch.group(2)
print('Day = ', day)
year = dmatch.group(3)
print('Year = ', year)

The article URL is extracted from the entry's anchor element.

In [ ]:
url = articles[0].find("div", "item-info").h2.a.get('href')
url

'https://www.npr.org/2023/07/09/1186694715/how-audiobooks-are-made'

In [ ]:
# Extract the ISO-formatted publication date from the first article


date2 = articles[0].find("p", "teaser").time.get('datetime')
date2

'2023-07-09'

In [ ]:
# Extract the third article's title

title3 = articles[2].find("div", "item-info").h2.text
title3

"Suspended from Twitter, the account tracking Elon Musk's jet has landed on Threads"

In [ ]:
# Extract the third article's teaser

teaser3 = (articles[2].find("div", "item-info").p.text).replace((articles[2].find("div", "item-info").p.span.text), ' ')
teaser3



" The account ElonJet tracked the movement of Elon Musk's private jet in real time, until it was suspended by Twitter last year. Now, it has resurfaced on Meta's fast-growing Twitter rival, Threads."

In [ ]:
# Extract the third article's publication date

date3 = articles[2].find("div", "item-info").p.span.text
date3

dmatch3 = re.match(r'([a-zA-z]+) (\d+), (\d+)', date3 )
month = dmatch3.group(1)
day = dmatch3.group(2)
year = dmatch3.group(3)

print(month, day, year)


July 9 2023


In [ ]:
# Extract the third article's URL

url3 = articles[2]. find("div", "item-info").h2.a.get("href")
url3


'https://www.npr.org/2023/07/09/1186680246/elon-musk-private-jet-twitter-threads'

In [ ]:
# Search technology headlines and teasers for TikTok
#nprtech['Title'].str.contains("TikTok") | nprtech['Teaser'].str.contains("TilTok")]

# Search technology headlines and teasers for Apple

def findInfo(article):

  try:
    title = article.find("div", "item-info").h2.text
    teaser = article.find("div", "item-info").p.text.replace((article.find("div", "item-info").p.span.text), '')
  except:
    return None

  return {
      'Title': title,
      'Teaser': teaser
  }

infos = [ findInfo(art) for art in articles if findInfo(art) is not None ]

target = 'Apple'

statuses = [ target in i['Title'] or target in i['Teaser'] for i in infos ]

mentions = sum(statuses)

if mentions == 0:
  print(f'No articles on {target}.')
else:
  print(f'There are {mentions} articles on {target}.')

No articles on Apple.


## Parse all technology articles

A reusable function applies the extraction logic to every valid article entry while skipping audio-only modules.

In [ ]:
def isaudio(article):
    """isaudio(article)
    Some "articles" are not true articles but just add buttons
    to the previously listed article to play an audio version
    of the story
    Input: article
    Output: returns True if the article is audio buttons, False
    otherwise"""

    return article.get('class')[0] == 'bucketwrap'

def articleinfo(article):
    """title, teaser, date, url = articleinfo(article)
    articleinfo - pulls information from an article's html
    Input: article
    Outputs: title, teaser, date, url"""

    # Skip audio-module "articles" which are not true articles
    # but just add buttons to play audio

    if not isaudio(article):
        title = article.find("div", "item-info").h2.text
        teaser = article.find("div", "item-info").p.text
        date = article.find("div", "item-info").p.span.text
        teaser = teaser.replace(date, '')
        url = article.find("div", "item-info").h2.a.get('href')
    else:
        title = []
        teaser = []
        date = []
        url = []

    return (title, teaser, date, url)


In [ ]:
titles = []
teasers = []
dates = []
urls = []

for article in articles:
    title, teaser, date, url = articleinfo(article)
    if title != []:
        titles.append(title)
        teasers.append(teaser)
        dates.append(date)
        urls.append(url)


In [ ]:
# Parse publication dates into month, day, and year
months = []
days = []
years = []
for date in dates:
    dmatch = re.match(r'([a-zA-Z]+) (\d+), (\d+)', date)
    months.append(dmatch.group(1))
    days.append(dmatch.group(2))
    years.append(dmatch.group(3))

The extracted titles, teasers, dates, and URLs are assembled into a pandas DataFrame.

In [ ]:
nprtech = pd.DataFrame({'Title': titles, 'Teaser': teasers, 'Month': months,
                       'Day': days, 'Year': years, 'url': urls})
nprtech.head()



,Title,Teaser,Month,Day,Year,url
0,The U.S. once built a nuclear ship ... for pas...,"In the Port of Baltimore, a ship is docked tha...",July,10,2023,https://www.npr.org/2023/07/06/1186209912/the-...
1,How audiobooks are made,From the podcast Nerdette from WBEZ: How can a...,July,9,2023,https://www.npr.org/2023/07/09/1186694715/how-...
2,"Suspended from Twitter, the account tracking E...",The account ElonJet tracked the movement of El...,July,9,2023,https://www.npr.org/2023/07/09/1186680246/elon...
3,Is there life after Twitter? A rundown of all ...,NPR's Ayesha Rascoe speaks with Washington Pos...,July,9,2023,https://www.npr.org/2023/07/09/1186674916/is-t...
4,The Sunday Story: Permission to share,"""It's this version of me that my mom's publici...",July,9,2023,https://www.npr.org/2023/07/06/1186221489/the-...


In [ ]:
# Repeat the extraction workflow for NPR's science archive
# https://www.npr.org/sections/science/archive
# Store the results in a DataFrame and combine both archive datasets

# Retrieve and parse the science archive HTML


url2 = 'https://www.npr.org/sections/science/archive'
html2 = requests.get(url2).text

soup2 = BeautifulSoup(html2, 'html5lib')

print(soup2.prettify())



In [ ]:
# Collect all science article elements

articles2 = soup2.find_all('article')
len(articles2)

24

In [ ]:
# Extract science article metadata

titles2 = []
teasers2 = []
dates2 = []
urls2 = []

for article in articles2:
  title, teaser, date, url = articleinfo(article)
  if title != []:
    titles2.append(title)
    teasers2.append(teaser)
    dates2.append(date)
    urls2.append(url)

In [ ]:
# Parse science publication dates

months2 = []
days2 = []
years2 = []

for date in dates2:
  dmatch = re.match(r'([a-zA-Z]+) (\d+), (\d+)', date)
  months2.append(dmatch.group(1))
  days2.append(dmatch.group(2))
  years2.append(dmatch.group(3))

In [ ]:
# Build the science article DataFrame

nprsci = pd.DataFrame({'Title': titles2, 'Teaser': teasers2, 'Month': months2, 'Day': days2, 'Year': years2, 'url': urls2 })

nprsci.head()


,Title,Teaser,Month,Day,Year,url
0,The U.S. once built a nuclear ship ... for pas...,"In the Port of Baltimore, a ship is docked tha...",July,10,2023,https://www.npr.org/2023/07/06/1186209912/the-...
1,The northern lights are coming to several stat...,This week's geomagnetic storm will bring the a...,July,9,2023,https://www.npr.org/2023/07/09/1186525577/nort...
2,Studying the link between the gut and mental h...,Calliope Holingue researches how the microbiom...,July,8,2023,https://www.npr.org/sections/health-shots/2023...
3,The world is officially 'free' of chemical wea...,The U.S. has destroyed the last of its stockpi...,July,7,2023,https://www.npr.org/2023/07/07/1186550955/the-...
4,Macron is far from the first leader to blame v...,Violence erupted in France following the fatal...,July,7,2023,https://www.npr.org/2023/07/07/1186531906/macr...


In [ ]:
 # Combine the technology and science datasets
nprTechSci =  nprsci.append(nprtech, ignore_index=True)

nprTechSci.head()

<ipython-input-26-08f8d0d0d016>:2: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  nprTechSci =  nprsci.append(nprtech, ignore_index=True)


,Title,Teaser,Month,Day,Year,url
0,The U.S. once built a nuclear ship ... for pas...,"In the Port of Baltimore, a ship is docked tha...",July,10,2023,https://www.npr.org/2023/07/06/1186209912/the-...
1,The northern lights are coming to several stat...,This week's geomagnetic storm will bring the a...,July,9,2023,https://www.npr.org/2023/07/09/1186525577/nort...
2,Studying the link between the gut and mental h...,Calliope Holingue researches how the microbiom...,July,8,2023,https://www.npr.org/sections/health-shots/2023...
3,The world is officially 'free' of chemical wea...,The U.S. has destroyed the last of its stockpi...,July,7,2023,https://www.npr.org/2023/07/07/1186550955/the-...
4,Macron is far from the first leader to blame v...,Violence erupted in France following the fatal...,July,7,2023,https://www.npr.org/2023/07/07/1186531906/macr...
